# Getting started with data




## Setting up the cluster

### Create a cluster

- We create a demo cluster with 1 control plane and 3 worker nodes using k3d.
- k3d is a lightweight wrapper that runs k3s (a minimal Kubernetes distribution) inside Docker containers, so you can spin up local multi-node Kubernetes clusters in seconds.

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# WINDOWS (Git Bash / WSL kernel): swap the line above for your install folder, e.g.
#   export PATH="/c/Program Files/Docker/Docker/resources/bin:$PATH"
# Host port 8081 -> load balancer port 80 (Traefik), so http://k8s.local:8081 reaches the ingress
if k3d cluster list k8scluster >/dev/null 2>&1; then
  echo "Cluster 'k8scluster' already exists, skipping creation."
else
  k3d cluster create k8scluster \
    --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
    --servers 1 \
    --agents 3 \
    --port "8081:80@loadbalancer"
fi

![k3d](images/k3d%20cluster.gif)

In [ ]:
%%bash
kubectl config get-contexts

### Update hostnames

In [ ]:
import subprocess
import getpass

hosts = [
    "k8s.local"
]

with open("/etc/hosts", "r") as f:
    current = f.read()

missing = [host for host in hosts if host not in current]
for host in hosts:
    if host not in missing:
        print(f"Already present: {host}")

if missing:
    password = getpass.getpass("sudo password: ")
    for host in missing:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} - {result.stderr.strip()}")

## Configurations

Containers should be built once and configured per environment. Kubernetes keeps configuration out of the image and <u>injects it at runtime</u>

- **ConfigMaps**: non-sensitive configuration (files, key/value pairs), consumed as a volume mount or as environment variables
- **Secrets**: sensitive values such as passwords and tokens, consumed the same ways. Stored base64-encoded, which is encoding, not encryption

### ConfigMaps

A ConfigMap stores non-sensitive configuration as key-value pairs or whole files, separate from the container image. A Pod can consume it in two ways:

- **Volume mount**: each key appears as a file in a directory inside the container. Mounted files update in place when the ConfigMap changes
- **Environment variables**: keys are injected as variables when the container starts. They are only read at startup, so a restart is needed to pick up changes

#### Creating a <u>Volume Mount</u>

In [ ]:
%%bash
kubectl create namespace k8s-ns

In [ ]:
%%bash
kubectl create cm k8s-cm -n k8s-ns --from-file=files/index.html

In [ ]:
%%bash
kubectl describe cm k8s-cm -n k8s-ns

#### Deploying Object

In [ ]:
%%bash
kubectl apply -f manifest/1-deploy_objects.yml
kubectl rollout status deploy/k8s-deployment -n k8s-ns

In [ ]:
%%bash
kubectl exec deploy/k8s-deployment -n k8s-ns  -- sh -c "cat /usr/share/nginx/html/index.html"

#### Creating <u>Environment Variables</u>

In [ ]:
%%bash
kubectl exec deploy/k8s-deployment -n k8s-ns -- env

In [ ]:
%%bash
kubectl apply -f manifest/2-create_env_configmap.yml
kubectl rollout status deploy/k8s-deployment -n k8s-ns

In [ ]:
%%bash
kubectl exec deploy/k8s-deployment -n k8s-ns -- env

## Storage

| Object | What it is | Analogy | Scope |
|---|---|---|---|
| **StorageClass** | The *type* of storage on offer (e.g. `local-path`). Names the provisioner that creates the storage, what happens to the data when you are done with it, and when the storage gets created. Holds no data itself. | A menu of options | Cluster |
| **PersistentVolumeClaim (PVC)** | The app's *request* for storage: "I need 1Gi that one node can read and write". Pods mount the claim, never the PV directly. | An order slip | Namespaced |
| **PersistentVolume (PV)** | The *actual piece of storage* that fulfils a claim, the place on a disk where the data really lives. | The dish that arrives | Cluster |

The app only asks for a claim. The StorageClass decides whether the PV is created for you (dynamic) or has to exist already (static).

![Storage](images/storage.gif)

In [ ]:
%%bash
kubectl get sc -o wide

In [ ]:
%%bash
kubectl get pods -n kube-system

In [ ]:
%%bash
kubectl describe sc local-path

### Dynamic vs static provisioning

`manifest/7-create_storage.yml` is file 6 plus:

| Resource | Kind | What it does | Mount path |
|---|---|---|---|
| **`k8s-pvc-dynamic`** | PVC | Names no StorageClass, so the default (`local-path`) creates the PV for us | `/data/dynamic` |
| **`manual`** | StorageClass | `kubernetes.io/no-provisioner`, `Retain`: nothing is created automatically | - |
| **PV we write by hand** | PV | The storage that the `manual` class points at | - |
| **`k8s-pvc-manual`** | PVC | Binds to the hand-written PV by class name | `/data/manual` |
| **`k8s-deployment`** (existing) | Deployment | Mounts both claims | both |

The deployment drops to **1 replica**: both volumes are `ReadWriteOnce` and node-local, so they cannot be shared across nodes.

In [ ]:
%%bash
kubectl apply -f manifest/3-create_storage.yml
kubectl rollout status deploy/k8s-deployment -n k8s-ns

In [ ]:
%%bash
kubectl get sc -o wide
echo ""
kubectl get pv
echo ""
kubectl get pvc -n k8s-ns
echo ""
kubectl describe pvc k8s-pvc-dynamic -n k8s-ns | grep -A8 Events

#### Copy a file from your machine into the volumes

In [ ]:
%%bash
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
echo "POD name: $POD"
# Copy hello.txt to the dynamic and manual volumes in the pod

echo "my dynamic content" > files/hello.txt
kubectl cp files/hello.txt k8s-ns/$POD:/data/dynamic/hello.txt

echo "my manual content" > files/hello.txt
kubectl cp files/hello.txt k8s-ns/$POD:/data/manual/hello.txt

# Verify the contents of the copied files
kubectl exec $POD -n k8s-ns -- cat /data/dynamic/hello.txt
kubectl exec $POD -n k8s-ns -- cat /data/manual/hello.txt

#### The volumes outlive the pod

Delete the pod. The Deployment creates a replacement, and the files are still there.

In [ ]:
%%bash
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl delete pod $POD -n k8s-ns
kubectl rollout status deploy/k8s-deployment -n k8s-ns

POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl exec $POD -n k8s-ns -- cat /data/dynamic/hello.txt
kubectl exec $POD -n k8s-ns -- cat /data/manual/hello.txt

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
docker exec k3d-k8scluster-agent-0 ls -l /data

In [ ]:
%%bash
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl exec $POD -n k8s-ns -- sh -c "ls -l /data"

#### Reclaim policy: `Delete` vs `Retain`

A claim in use is protected from deletion, so scale the Deployment to zero first. Then delete both claims:

- `local-path` (`Delete`): the PV and its data are removed
- `manual` (`Retain`): the PV stays in `Released` and the data stays on the node. A released PV is not reused automatically, an administrator must clear its `claimRef` or delete it

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# WINDOWS (Git Bash / WSL kernel): swap the line above for your install folder, e.g.
#   export PATH="/c/Program Files/Docker/Docker/resources/bin:$PATH"
DYN=$(kubectl get pvc k8s-pvc-dynamic -n k8s-ns -o jsonpath='{.spec.volumeName}')
kubectl scale deploy/k8s-deployment --replicas=0 -n k8s-ns
kubectl wait --for=delete pod -l app=k8s-deployment -n k8s-ns --timeout=60s
kubectl delete pvc k8s-pvc-dynamic k8s-pvc-manual -n k8s-ns
kubectl wait --for=delete pv/$DYN --timeout=60s

echo "== local-path PV is gone, manual PV is Released"
kubectl get pv

echo "== The manual data is still on the node"
docker exec k3d-k8scluster-agent-0 cat /data/k8s-pv-manual/hello.txt

## Cleanup

### Clean up storage

Remove the manual PV and class, then re-apply file 6 to put the Deployment back to 3 replicas without volumes.

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# WINDOWS (Git Bash / WSL kernel): swap the line above for your install folder, e.g.
#   export PATH="/c/Program Files/Docker/Docker/resources/bin:$PATH"
kubectl delete pv k8s-pv-manual --ignore-not-found
kubectl delete sc manual --ignore-not-found
docker exec k3d-k8scluster-agent-0 rm -rf /data/k8s-pv-manual

kubectl delete -f manifest/3-create_env_configmap.yml
kubectl rollout status deploy/k8s-deployment -n k8s-ns

kubectl get sc
kubectl get pv

### Delete Cluster

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# WINDOWS (Git Bash / WSL kernel): swap the line above for your install folder, e.g.
#   export PATH="/c/Program Files/Docker/Docker/resources/bin:$PATH"
# Delete the entire cluster - removes all Docker containers and volumes
k3d cluster delete k8scluster

### Reclaim Disk Space

In [ ]:
%%bash
docker system df
echo ""
echo "=== Dangling volumes ==="
docker volume ls --filter dangling=true
echo ""
docker volume prune -f

### Delete Host Name

**macOS / Linux**

In [ ]:
import subprocess
import getpass

hosts = [
    "k8s.local"
]

with open("/etc/hosts", "r") as f:
    lines = f.readlines()

present = [host for host in hosts if any(host in line.split() for line in lines)]
for host in hosts:
    if host not in present:
        print(f"Not present: {host}")

if present:
    password = getpass.getpass("sudo password: ")
    kept = [line for line in lines if not any(host in line.split() for host in present)]
    result = subprocess.run(
        ["sudo", "-S", "tee", "/etc/hosts"],
        input=f"{password}\n{''.join(kept)}",
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        for host in present:
            print(f"Removed: {host}")
    else:
        print(f"Failed: {result.stderr.strip()}")